In [1]:
import glob

import cv2
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import patches as mpatches

In [2]:
titles = {
    'L2': r'$L_2$',
    'SSIM': 'SSIM',
    'MONAI_alex': 'AlexNet',
    'MONAI_vgg': 'VGG-16',
    'UltraPIPS_vit_imagenet': 'ViT',
    'UltraPIPS_swin_imagenet': 'SwinViT',
    'UltraPIPS_clip': 'CLIP',
    'MONAI_radimagenet_resnet50': 'RadImageNet',
    'UltraPIPS_medsam': 'MedSAM',
    'UltraPIPS_biomedclip': 'BiomedCLIP',
    'UltraPIPS_usfm': 'USFM',
    'UltraPIPS_tusa_vit': 'TUSA',
    'UltraPIPS_ultrasound_clip': 'Ultrasound-CLIP',
}

order = [
    r'$L_2$', 'SSIM',
    'AlexNet', 'VGG-16', 'ViT', 'SwinViT', 'CLIP',
    'RadImageNet', 'MedSAM', 'BiomedCLIP',
    'USFM', 'TUSA', 'Ultrasound-CLIP'
]

groups = {
    'Classical': [r'$L_2$', 'SSIM'],
    'ImageNet': ['AlexNet', 'VGG-16', 'ViT', 'SwinViT', 'CLIP'],
    'Radiology': ['RadImageNet', 'MedSAM', 'BiomedCLIP'],
    'Ultrasound': ['USFM', 'TUSA', 'Ultrasound-CLIP']
}

colors = {
    'Classical': '#34495e',
    'ImageNet': '#3498db',
    'Radiology': '#e67e22',
    'Ultrasound':  '#2ecc71'
}


In [3]:
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
all_metrics = {g: [] for g in groups}
for res_file in glob.glob('../assets/results/01/rotation/*/A2C.npz'):
    data = np.load(res_file)
    metrics = [k for k in data.keys() if k not in ['y', 'yhat', 'confidence', 'variants']]

    for metric in metrics:
        group = [g for g, l in groups.items() if titles[metric] in l][0]
        all_metrics[group].append(data[metric][3, :-1] / np.linalg.norm(data[metric][3, :-1]))

plt.figure(figsize=(5, 5))
y, s = {}, {}
for group, losses in all_metrics.items():
    y[group] = np.vstack(losses).mean(axis=0)
    s[group] = np.vstack(losses).std(axis=0)
    
color_list = [colors[g] for g in y]
for i in range(5):
    plt.bar(
        range(len(y)),
        [group_y[i] for group_y in y.values()],
        color=color_list,
        yerr=[group_s[i] for group_s in s.values()],
        ecolor=color_list
    )

    plt.xticks([])
    plt.ylim([0, 1])
    
    if i == 0:
        plt.ylabel('Normalized Distance', fontsize=24)
        plt.yticks(np.linspace(0, 1, 10), labels=[0] + [None]*8 + [1], fontsize=24)
    else:
        plt.yticks(np.linspace(0, 1, 10), labels=[None]*10)



    plt.tight_layout()
    plt.savefig(f'../assets/results/03/models_{i}.png')

In [5]:
def get_masks(lv, la):
    lv_mask = cv2.imread(lv, cv2.IMREAD_GRAYSCALE)
    la_mask = cv2.imread(la, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(lv_mask | la_mask, (256, 256), interpolation=cv2.INTER_AREA)
    return mask

In [6]:
from echogains.utils import get_repaint_mask

In [7]:
im = cv2.imread(f'../assets/results/03/sam_figure/A2C_batch_0_frame003/base_image.png', cv2.IMREAD_GRAYSCALE)
mask = get_repaint_mask(im)

In [9]:
for rotation in range(5, 30, 5):
    gt = get_masks(
        f'../assets/results/03/sam_figure/A2C_batch_0_frame003/rotation_{rotation:.1f}_gt_lv.png',
        f'../assets/results/03/sam_figure/A2C_batch_0_frame003/rotation_{rotation:.1f}_gt_la.png'
    ) * mask
    pred = get_masks(
        f'../assets/results/03/sam_figure/A2C_batch_0_frame003/rotation_{rotation:.1f}_pred_lv.png',
        f'../assets/results/03/sam_figure/A2C_batch_0_frame003/rotation_{rotation:.1f}_pred_la.png'
    ) # * mask

    g, p = gt > 0, pred > 0
    im = cv2.cvtColor(np.dstack((p & ~g, g & p, g & ~p)).astype(np.uint8) * 255, cv2.COLOR_RGB2BGR)
    im[(gt == 0) & (pred == 0)] = 0
    im[(gt == 0) & (pred == 0) & (mask > 0)] = 200
    cv2.imwrite(f'../assets/results/03/ultrasam_rotation_{rotation}.png', im)

In [ ]:
categories = {
    'True Positive': [0, 1, 0],
    'False Positive': [1, 0, 0],
    'False Negative': [0, 0, 1],
    'Ultrasound Sector': [200/255., 200/255., 200/255.]
}

fig_leg = plt.figure(figsize=(8, 0.5))
legend_handles = [mpatches.Patch(color=c, label=cat) for cat, c in categories.items()]
fig_leg.legend(handles=legend_handles, loc='center', ncol=4, frameon=False, fontsize=10)
plt.savefig('../assets/results/03/dsclegend.png', bbox_inches='tight', transparent=True)